<h1>Project 3</h1>
<h3>Harry Hawkins 48019675 - 2024</h3>
<h4><a href="https://github.com/Unicron0486/Harry-Hawkins-2504-2024-PROJECT3/tree/main">Git repo here</h4>

In [3]:
using Pkg
Pkg.activate(".")

  Activating project at `C:\Users\hawki\Uni\M2504\P3\Harry-Hawkins-2504-2024-PROJECT3`


<h2>General data observations</h2>

In [10]:
using CSV, DataFrames, HTTP, CSV, Plots, StatsPlots, StatsBase
path = "./data/Melbourne_housing_FULL.csv"
old_df = CSV.read(path, DataFrame)

#Propertycount: Number of properties that exist in the suburb.

# Distance: Distance from CBD in Kilometres

# Method:
# S - property sold;
# SP - property sold prior;
# PI - property passed in;
# PN - sold prior not disclosed;
# SN - sold not disclosed;
# NB - no bid;
# VB - vendor bid;
# W - withdrawn prior to auction;
# SA - sold after auction;
# SS - sold after auction price not disclosed.
# N/A - price or highest bid not available.

# Type:
# br - bedroom(s);
# h - house,cottage,villa, semi,terrace;
# u - unit, duplex;
# t - townhouse;
# dev site - development site;
# o res - other residential.

# SellerG: Real Estate Agent

# Bedroom2 : Scraped # of Bedrooms (from different source)
println("collected data")

collected data


<p>For easier interaction I will change data types of some cloumns. 
Checking for missing values will allow me to set an appropriate missing value for the new type. 
If the data type is numeric -1 will be replacing missing values, with the exception of longitude and latitude which will be a 0.</p>

In [17]:
df = copy(old_df)
# unique(df.Suburb) #No missing
# unique(df.Address) #No missing
# unique(df.Rooms) #No missing
# unique(df.Type) #No missing
# unique(df.Method) #No missing
# unique(df.SellerG) #No missing
# unique(df.Date) #No missing

# unique(df.Price) #missing as missing => will need to skip or avg
df.Price .= coalesce.(df.Price, -1)


# unique(df.Distance) #missin as #N/A
for (i,n) in enumerate(df.Distance)
    if n == "#N/A"
        df.Distance[i] = "-1"
    end
end
df.Distance .= parse.(Float64, df.Distance)

# unique(df.Landsize) #missing as missing => -1
df.Landsize .= coalesce.(df.Landsize, -1)

# unique(df.BuildingArea) #missing as missing => -1
df.BuildingArea .= coalesce.(df.BuildingArea, -1)


# unique(df.YearBuilt) #missing as missing => -1
df.YearBuilt .= coalesce.(df.YearBuilt, -1)

# unique(df.Lattitude) #missing as missing => ?
# unique(typeof.(df.Longtitude)) #missing as missing => ?
df.Lattitude .= coalesce.(df.Lattitude, 0.0)
df.Longtitude .= coalesce.(df.Longtitude, 0.0)


# unique(df.Bedroom2) #missing as missing => keep as
# unique(df.Bathroom) #missing as missing => keep as
# unique(df.Car) #missing as missing => keep as

allowmissing!(df, :CouncilArea)
allowmissing!(df,:Postcode)
allowmissing!(df,:Regionname)
allowmissing!(df, :Propertycount)


# unique(df.CouncilArea) #missing as #N/A => changing based on suburb/address or down as missing
function findarea(suburb::String31)
    for (i,n) in enumerate(df.Suburb)
        if n == suburb && df.CouncilArea[i] != "#N/A"
            return df.CouncilArea[i]
        end
    end
    return missing
end


for (i,n) in enumerate(df.CouncilArea)
    if n == "#N/A" && !(df.Suburb[i] isa Nothing)
        df.CouncilArea[i] = findarea(df.Suburb[i])
    end
end


# unique(df.Postcode) #missing as #N/A => changing based on suburb/address
function findcode(suburb::String31)
    for (i,n) in enumerate(df.Suburb)
        if n == suburb && df.Postcode[i] != "#N/A"
            return df.Postcode[i]
        end
    end
    return missing
end

for (i,n) in enumerate(df.Postcode)
    if n == "#N/A"
        df.Postcode[i] = findcode(df.Suburb[i])
    end
end

# unique(df.Regionname) #missing as #N/A => changing based on suburb/address
function findregion(suburb::String31)
    for (i,n) in enumerate(df.Suburb)
        if n == "suburb" && df.Regionname[i] != "#N/A"
            return df.Regionname[i]
        end
    end
    return missing
end
for (i,n) in enumerate(df.Regionname)
    if n == "#N/A"
        df.Regionname[i] = findregion(df.Suburb[i])
    end
end

# unique(df.Propertycount) #missing as #N/A => changing based on suburb/address
function findcount(suburb::String31)
    for (i,n) in enumerate(df.Suburb)
        if n == "suburb" && df.Propertycount[i] != "#N/A"
            return df.Propertycount[i]
        end
    end
    return missing
end

for (i,n) in enumerate(df.Propertycount)
    if n == "#N/A"
        df.Propertycount[i] = findcount(df.Suburb[i])
    end
end

df.Propertycount = passmissing(parse).(Int64, df.Propertycount)
df.Postcode .= passmissing(parse).(Int64, df.Postcode)

# turns Date string into a form that can be turned into a Date type
function parsedate(date)
    result = ""
    for n in date
        if n == '/'
            result *= "-"
        else
            result *= n
        end
    end
    return result
end

#Creates the Date column to be a Date type
insertcols!(df, :newDate => Date("0001-01-01"))
for (i,n) in enumerate(df.Date)
    date = String(parsedate(n))
    df.newDate[i] = Date(date, "d-m-y")
end
df = select!(df, Not(:Date))
rename!(df, :newDate => :Date)


println("Missing values & data types changed")

Missing values & data types changed


<h2>Task 1</h2>
<p>plots of Rooms, Price, Method, Distance, Landsize</p>

In [ ]:
typeof(Rooms)
typeof

<h2>Task 2</h2>
<p>plots of house price with respect to distance, size(land, room,car),, and combinations of the formentioned vars. Search for relation ship</p>

<h2>Task 3</h2>
<p>plots that help visualise trends in data. volume of sales (num props and values) over time. also type h over time. time in months</p>

<h2>Task 4</h2>
<p>review task 2 and fit linear regression for predicting house prices as a function of all variales. Asses th quality of the model by using training and validation.</p>